In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


df = pd.read_csv('Project_description_and_data/claims_train.csv')
df_test = pd.read_csv('Project_description_and_data/claims_test.csv')


def basic_feature_engineering(df):
    df = df.copy()
    
    df['Exposure_adj'] = df['Exposure'].clip(upper=1.0)
    df['VehAge_1'] = df['VehAge'] + 1
    df['VehAge_log'] = np.log(df['VehAge_1'])
    df['DrivAge_log'] = np.log(df['DrivAge'])
    df['Density_log'] = np.log(df['Density'])
    
    cat_cols = ['VehBrand', 'Area', 'Region', 'VehGas']
    for c in cat_cols:
        df[c] = df[c].astype('category')
        
    return df


df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)

df_train = basic_feature_engineering(df_train)
df_val   = basic_feature_engineering(df_val)
df_test  = basic_feature_engineering(df_test)

pca_features = ['DrivAge_log', 'VehPower', 'VehAge_log', 'Density_log', 'BonusMalus']

scaler_pca = StandardScaler()
X_pca_train = scaler_pca.fit_transform(df_train[pca_features])

pca = PCA(n_components=3, random_state=42)
X_pca_train = pca.fit_transform(X_pca_train)

for i in range(3):
    df_train[f'PC{i+1}'] = X_pca_train[:, i]

def apply_pca(df, scaler_pca, pca, pca_features):
    X_scaled = scaler_pca.transform(df[pca_features])
    X_pca = pca.transform(X_scaled)
    
    df = df.copy()
    for i in range(X_pca.shape[1]):
        df[f'PC{i+1}'] = X_pca[:, i]
    return df

df_val  = apply_pca(df_val, scaler_pca, pca, pca_features)
df_test = apply_pca(df_test, scaler_pca, pca, pca_features)

def add_interactions(df):
    df = df.copy()
    df['DrivAgeLog_BonusMalus'] = df['DrivAge_log'] * df['BonusMalus']
    df['VehPower_DensityLog']  = df['VehPower'] * df['Density_log']
    df['BonusMalus_Exposure']  = df['BonusMalus'] * df['Exposure_adj']
    return df

df_train = add_interactions(df_train)
df_val   = add_interactions(df_val)
df_test  = add_interactions(df_test)

feature_cols = ['VehBrand','Region','PC1', 'PC2', 'PC3','DrivAgeLog_BonusMalus','VehPower_DensityLog','BonusMalus_Exposure']

categorical_features = ['VehBrand', 'Region']
numerical_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         categorical_features),
        ('num', 'passthrough', numerical_features)])

preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(df_train[feature_cols])
X_val   = preprocessor.transform(df_val[feature_cols])
X_test  = preprocessor.transform(df_test[feature_cols])

X_train.columns = [c.split("__")[-1] for c in X_train.columns]
X_val.columns   = X_train.columns
X_test.columns  = X_train.columns

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

y_train = df_train['ClaimNb'].values.reshape(-1, 1)
y_val   = df_val['ClaimNb'].values.reshape(-1, 1)
y_test  = df_test['ClaimNb'].values.reshape(-1, 1)


def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

class FFNNRegressor:

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate=0.01, 
                 l2_lambda=0.0, dropout_rate=0.0):
        
        self.lr = learning_rate
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)     # weights initialisation (He)
        self.b1 = np.zeros((1, hidden_dim))
        
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)       # output dim=1 (predicting 1 value)
        self.b2 = np.zeros((1, output_dim))
        
    def forward(self, X, training=True):
        self.z1 = X @ self.W1 + self.b1         #hidden layer
        self.a1 = relu(self.z1)
        
        if training and self.dropout_rate > 0:      #applying dropout
            keep_prob = 1 - self.dropout_rate
            self.dropout_mask = np.random.binomial(1, keep_prob, size=self.a1.shape)
            self.a1 = self.a1 * self.dropout_mask / keep_prob
        
        self.z2 = self.a1 @ self.W2 + self.b2       # Output layer
        self.y_pred = self.z2 
        
        return self.y_pred
    
    def backward(self, X, y_true):
        batch_size = X.shape[0]

        dz2 = self.y_pred - y_true      # gradient calculation with MSE
        
        self.dW2 = (self.a1.T @ dz2) / batch_size
        self.db2 = np.sum(dz2, axis=0, keepdims=True) / batch_size
        
        if self.l2_lambda > 0:                          #L2 regularization gradient for W2
            self.dW2 += self.l2_lambda * self.W2
        
        da1 = dz2 @ self.W2.T                           #hidden layer gradients
        
        if self.dropout_rate > 0:                       #dropout mask to gradient
            keep_prob = 1 - self.dropout_rate
            da1 = da1 * self.dropout_mask / keep_prob
        
        dz1 = da1 * relu_derivative(self.z1)
        self.dW1 = (X.T @ dz1) / batch_size
        self.db1 = np.sum(dz1, axis=0, keepdims=True) / batch_size
        
        if self.l2_lambda > 0:                  #L2 regularization gradient for W1
            self.dW1 += self.l2_lambda * self.W1
        
    def update_weights(self):                   #with gradient descent
        self.W1 -= self.lr * self.dW1
        self.b1 -= self.lr * self.db1
        self.W2 -= self.lr * self.dW2
        self.b2 -= self.lr * self.db2
        
    def compute_loss(self, y_true, y_pred):
        batch_size = y_true.shape[0]
        
        mse_loss = np.mean(0.5 * (y_pred - y_true)**2)
        
        if self.l2_lambda > 0:
            l2_loss = (self.l2_lambda / 2) * (np.sum(self.W1**2) + np.sum(self.W2**2))
            return mse_loss + l2_loss
        return mse_loss
    
    def predict(self, X):
        return self.forward(X, training=False)
    
    def evaluate_metric(self, X, y_true):
        y_pred = self.predict(X)
        return r2_score(y_true, y_pred)
    
def train_model(model, X_train, y_train, X_val, y_val, num_epochs, verbose=False):
    train_losses = []
    val_losses = []
    train_scores = []
    val_scores = []
    
    for epoch in range(num_epochs):
        y_pred_train = model.forward(X_train, training=True)
        train_loss = model.compute_loss(y_train, y_pred_train)
        model.backward(X_train, y_train)
        model.update_weights()
        
        y_pred_val = model.forward(X_val, training=False)
        val_loss = model.compute_loss(y_val, y_pred_val)
        
        train_score = model.evaluate_metric(X_train, y_train)
        val_score = model.evaluate_metric(X_val, y_val)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_scores.append(train_score)
        val_scores.append(val_score)
        
        if verbose and (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val R2: {val_score:.4f}")

    return train_losses, val_losses, train_scores, val_scores


lambda_values = [0.001, 0.01, 0.1]
l2_results = {}

for lam in lambda_values:
    print(f"Training with L2 λ = {lam}")
    
    model = FFNNRegressor(
        input_dim=X_train.shape[1],
        hidden_dim=128,
        output_dim=1,
        learning_rate=0.01,
        l2_lambda=lam
    )
    
    _, _, _, val_scores = train_model(
        model,
        X_train, y_train,
        X_val, y_val,
        num_epochs=250
    )
    
    l2_results[lam] = val_scores[-1]
    print(f"  Final Val R2: {val_scores[-1]:.4f}")

dropout_rates = [0.0, 0.2, 0.4]
dropout_results = {}

for d in dropout_rates:
    print(f"Training with dropout = {d}")
    
    model = FFNNRegressor(
        input_dim=X_train.shape[1],
        hidden_dim=128,
        output_dim=1,
        learning_rate=0.01,
        dropout_rate=d
    )
    
    _, _, _, val_scores = train_model(
        model,
        X_train, y_train,
        X_val, y_val,
        num_epochs=250
    )
    
    dropout_results[d] = val_scores[-1]
    print(f"  Final Val R2: {val_scores[-1]:.4f}")

best_lambda = max(l2_results, key=l2_results.get)
best_dropout = max(dropout_results, key=dropout_results.get)

print("\nBest hyperparameters (from validation only)")
print(f"Best L2 λ: {best_lambda}")
print(f"Best dropout: {best_dropout}")

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.vstack([y_train, y_val])

final_model = FFNNRegressor(
    input_dim=X_train.shape[1],
    hidden_dim=128,
    output_dim=1,
    learning_rate=0.01,
    l2_lambda=best_lambda,
    dropout_rate=best_dropout
)

print("\nTraining final model on train + validation data")

train_model(
    final_model,
    X_train_full, y_train_full,
    X_val, y_val,
    num_epochs=300,
    verbose=True
)

print("\nFINAL TEST SET EVALUATION")

y_test_pred = final_model.predict(X_test)

test_mse = mean_squared_error(y_test, y_test_pred)
test_r2  = r2_score(y_test, y_test_pred)
test_mae=mean_absolute_error(y_test, y_test_pred)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae: .4f}")
print(f"Test R2:  {test_r2:.4f}")

Training with L2 λ = 0.001
  Final Val R2: -1.4887
Training with L2 λ = 0.01
  Final Val R2: -1.0662
Training with L2 λ = 0.1
  Final Val R2: -0.4463
Training with dropout = 0.0
  Final Val R2: -1.1716
Training with dropout = 0.2
  Final Val R2: -0.3471
Training with dropout = 0.4
  Final Val R2: -0.0483

Best hyperparameters (from validation only)
Best L2 λ: 0.1
Best dropout: 0.4

Training final model on train + validation data
Epoch  50 | Train Loss: 11.7087 | Val Loss: 11.4307 | Val R2: -1.6426
Epoch 100 | Train Loss: 10.4093 | Val Loss: 10.2795 | Val R2: -0.4793
Epoch 150 | Train Loss: 9.3514 | Val Loss: 9.2805 | Val R2: -0.1945
Epoch 200 | Train Loss: 8.4329 | Val Loss: 8.3892 | Val R2: -0.0902
Epoch 250 | Train Loss: 7.6178 | Val Loss: 7.5879 | Val R2: -0.0442
Epoch 300 | Train Loss: 6.8873 | Val Loss: 6.8653 | Val R2: -0.0214

FINAL TEST SET EVALUATION
Test MSE: 0.0614
Test MAE:  0.1030
Test R2:  -0.0194
